In [1]:
import os

print("je m'execute depuis :", os.getcwd())
print("le dossier data existe :", os.path.isdir("../data"))
print("le fichier existe     :", os.path.isfile("../data/restaurants.json"))
print("contenu de ../data    :", os.listdir("../data"))

je m'execute depuis : /Users/kanellis/json-api-geocodage/notebooks
le dossier data existe : True
le fichier existe     : True
contenu de ../data    : ['restaurants.json']


In [2]:
import json

with open("../data/restaurants.json", encoding="utf-8") as f:
    data = json.load(f)

print(type(data))
print(data["ville"], "-", data["pays"])
print("nombre de restaurants :", len(data["restaurants"]))

<class 'dict'>
Catane - Italie
nombre de restaurants : 4


In [3]:
print(data["restaurants"][0]["nom"])
print(data["restaurants"][0]["notes"][0])

Gelateria Zio Pietro dal 1964
4.6


In [4]:
for r in data["restaurants"]:
    moyenne = sum(r["notes"]) / len(r["notes"])
    print(f"{r['nom']:32} {moyenne:.2f}")

Gelateria Zio Pietro dal 1964    4.70
A Casa d'Amici                   4.45
La Bitta                         4.15
Steak House                      4.00


In [5]:
print(json.dumps(data["restaurants"][1], ensure_ascii=False, indent=2))


{
  "nom": "A Casa d'Amici",
  "adresse": "Via Fischetti, 14, 95131 Catania CT",
  "type": "trattoria",
  "notes": [
    4.4,
    4.5
  ]
}


In [6]:
import pandas as pd

df = pd.json_normalize(data)
print(df.shape)
print(df.columns.tolist())

(1, 4)
['ville', 'pays', 'releve', 'restaurants']


In [7]:
df = pd.json_normalize(data, record_path="restaurants")
print(df)

                             nom                                   adresse  \
0  Gelateria Zio Pietro dal 1964  Via Porta di Ferro, 47, 95131 Catania CT   
1                 A Casa d'Amici       Via Fischetti, 14, 95131 Catania CT   
2                       La Bitta    Via Acquicella Porto, 95121 Catania CT   
3                    Steak House   Via Porta di Ferro, 8, 95100 Catania CT   

        type                 notes  
0    glacier       [4.6, 4.8, 4.7]  
1  trattoria            [4.4, 4.5]  
2    poisson  [4.2, 4.0, 4.3, 4.1]  
3      grill            [3.9, 4.1]  


In [8]:
df = pd.json_normalize(data, record_path="restaurants", meta=["ville", "pays"])
print(df)

                             nom                                   adresse  \
0  Gelateria Zio Pietro dal 1964  Via Porta di Ferro, 47, 95131 Catania CT   
1                 A Casa d'Amici       Via Fischetti, 14, 95131 Catania CT   
2                       La Bitta    Via Acquicella Porto, 95121 Catania CT   
3                    Steak House   Via Porta di Ferro, 8, 95100 Catania CT   

        type                 notes   ville    pays  
0    glacier       [4.6, 4.8, 4.7]  Catane  Italie  
1  trattoria            [4.4, 4.5]  Catane  Italie  
2    poisson  [4.2, 4.0, 4.3, 4.1]  Catane  Italie  
3      grill            [3.9, 4.1]  Catane  Italie  


In [9]:
print(len(villes))

NameError: name 'villes' is not defined

In [10]:
for ville in data["villes"]:
    print(f"Ville : {ville['nom']} | Nombre de restaurants : {len(ville['restaurants'])}")

KeyError: 'villes'

In [11]:
import json

# Ouvrir et charger le fichier JSON situé dans le dossier data
with open("../data/villes.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("Fichier JSON chargé avec succès !")

Fichier JSON chargé avec succès !


In [12]:
for ville in data["villes"]:
    print(f"Ville : {ville['nom']} | Nombre de restaurants : {len(ville['restaurants'])}")


Ville : Catane | Nombre de restaurants : 3
Ville : Palerme | Nombre de restaurants : 2
Ville : Syracuse | Nombre de restaurants : 2


In [13]:
import pandas as pd

df = pd.json_normalize(
    data,
    record_path=['villes', 'restaurants'],
    meta=[
        'pays',
        ['villes', 'nom'],
        ['villes', 'region'],
        ['villes', 'habitants']
    ]
)

# Nettoyage et renommage des colonnes issues des niveaux intermédiaires
df = df.rename(columns={
    'villes.nom': 'ville',
    'villes.region': 'région',
    'villes.habitants': 'habitants'
})

# Affichage des colonnes principales demandées
print(df[['nom', 'type', 'ville', 'région', 'habitants']])

                             nom       type     ville  région habitants
0  Gelateria Zio Pietro dal 1964    glacier    Catane  Sicile    298762
1                 A Casa d'Amici  trattoria    Catane  Sicile    298762
2                       La Bitta    poisson    Catane  Sicile    298762
3              Antica Focacceria  trattoria   Palerme  Sicile    630828
4               Ferro di Cavallo  trattoria   Palerme  Sicile    630828
5                    Don Camillo    poisson  Syracuse  Sicile    121605
6               Gelateria Bianca    glacier  Syracuse  Sicile    121605


In [14]:
# Calcul de la note moyenne
df['note_moyenne'] = df['notes'].apply(lambda x: round(sum(x) / len(x), 2))

# 1. Quelle ville a la meilleure note moyenne ?
meilleure_ville = df.groupby('ville')['note_moyenne'].mean().idxmax()
print(f"Ville avec la meilleure note moyenne : {meilleure_ville}")

# 2. Combien de restaurants de chaque type, tous lieux confondus ?
types_count = df['type'].value_counts()
print("\nNombre de restaurants par type :")
print(types_count)

Ville avec la meilleure note moyenne : Syracuse

Nombre de restaurants par type :
type
trattoria    3
glacier      2
poisson      2
Name: count, dtype: int64
